# pHash Evasion — Attack Visualization

**Author:** Avijit Roy — FDEPH Project  
**Purpose:** Show exactly what the attack does — original image, adversarial image,
difference map, and hash grids (original / adversarial / XOR of flipped bits).

This is the thesis figure that makes the attack tangible:
- The image barely changes (high SSIM, invisible difference)
- The hash changes just enough to cross the detection threshold
- The XOR grid shows exactly which bits flipped


In [ ]:
import os, sys
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("Repo root:", REPO_ROOT)

In [ ]:
import glob
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

from utils.phash_torch import compute_phash_hard
from utils.image_processing import load_and_preprocess_img

ADV_DIR = os.path.join(REPO_ROOT, "evasion_attack_outputs_phash")
ORI_DIR = os.path.join(REPO_ROOT, "inputs", "inputs_500")
FIG_DIR = os.path.join(REPO_ROOT, "fdeph_eval", "analysis", "figures")
os.makedirs(FIG_DIR, exist_ok=True)

device = torch.device("cpu")

def find_original(stem):
    for ext in ['.JPEG', '.jpeg', '.jpg', '.png']:
        p = os.path.join(ORI_DIR, stem + ext)
        if os.path.exists(p):
            return p
    return None

def bits_to_grid(hash_tensor, rows=8, cols=8):
    """Reshape 64-bit hash into 8x8 grid."""
    return hash_tensor.numpy().reshape(rows, cols)

print("Loaded.")

In [ ]:
def draw_hash_grid(ax, grid, title, flip_mask=None):
    """Draw an 8x8 binary hash as a colored grid.
    flip_mask: boolean 8x8 array marking flipped bits in red.
    """
    ax.set_xlim(0, 8)
    ax.set_ylim(0, 8)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title, fontsize=9, pad=4)

    for r in range(8):
        for c in range(8):
            bit = grid[r, c]
            if flip_mask is not None and flip_mask[r, c]:
                color = '#E24B4A'   # red = flipped
                edge  = '#A32D2D'
                lw    = 1.2
            elif bit == 1:
                color = '#378ADD'   # blue = bit 1
                edge  = '#185FA5'
                lw    = 0.4
            else:
                color = '#f0f0f0'   # light gray = bit 0
                edge  = '#cccccc'
                lw    = 0.4

            rect = patches.Rectangle(
                (c, 7 - r), 1, 1,
                linewidth=lw, edgecolor=edge, facecolor=color
            )
            ax.add_patch(rect)


def visualize_attack(ori_path, adv_path, ax_row, amp=10):
    """Fill one row of axes with: orig | adv | diff×amp | orig_hash | adv_hash | xor"""
    # Load images as PIL for display
    ori_pil = Image.open(ori_path).convert('RGB')
    adv_pil = Image.open(adv_path).convert('RGB')

    ori_np = np.array(ori_pil).astype(float) / 255.0
    adv_np = np.array(adv_pil).convert('RGB') if not isinstance(adv_pil, np.ndarray) else adv_pil
    adv_np = np.array(Image.open(adv_path).convert('RGB')).astype(float) / 255.0

    # Resize adv to match ori if needed
    if ori_np.shape != adv_np.shape:
        adv_pil_r = Image.open(adv_path).convert('RGB').resize(
            (ori_pil.width, ori_pil.height), Image.LANCZOS)
        adv_np = np.array(adv_pil_r).astype(float) / 255.0

    diff_np = np.clip(np.abs(ori_np - adv_np) * amp, 0, 1)

    # Compute pHash on tensors
    ori_t = load_and_preprocess_img(ori_path, device, resize=True)
    adv_t = load_and_preprocess_img(adv_path, device, resize=True)

    ori_hash, ori_hex, _ = compute_phash_hard(ori_t)
    adv_hash, adv_hex, _ = compute_phash_hard(adv_t)

    ori_grid = bits_to_grid(ori_hash)
    adv_grid = bits_to_grid(adv_hash)
    xor_grid = bits_to_grid((ori_hash != adv_hash).float())
    flip_mask = (ori_hash != adv_hash).numpy().reshape(8, 8)

    bits_flipped = int((ori_hash != adv_hash).sum().item())
    d_norm = bits_flipped / 64.0

    # Distortion metrics
    l_inf = float(np.max(np.abs(ori_np - adv_np)))

    # --- Plot ---
    stem = os.path.basename(adv_path).replace('_opt.png', '')

    ax_row[0].imshow(ori_np)
    ax_row[0].set_title('Original', fontsize=9)
    ax_row[0].axis('off')

    ax_row[1].imshow(adv_np)
    ax_row[1].set_title('Adversarial', fontsize=9)
    ax_row[1].axis('off')

    ax_row[2].imshow(diff_np)
    ax_row[2].set_title(f'Difference ×{amp}\nL∞={l_inf:.4f}', fontsize=9)
    ax_row[2].axis('off')

    draw_hash_grid(ax_row[3], ori_grid,
                   f'Original hash\n{ori_hex[:8]}...')

    draw_hash_grid(ax_row[4], adv_grid, flip_mask=flip_mask,
                   title=f'Adversarial hash\n{adv_hex[:8]}...')

    draw_hash_grid(ax_row[5], xor_grid, flip_mask=flip_mask,
                   title=f'Flipped bits (XOR)\n{bits_flipped}/64 = {d_norm:.3f}')

print("Functions defined.")

## Main visualization — 5 representative images

In [ ]:
# Find adversarial / original pairs
all_adv = sorted(glob.glob(os.path.join(ADV_DIR, "*_opt.png")))

pairs = []
for adv_path in all_adv:
    stem = os.path.basename(adv_path).replace('_opt.png', '')
    ori_path = find_original(stem)
    if ori_path:
        pairs.append((ori_path, adv_path))

print(f"Found {len(pairs)} original/adversarial pairs")

# Pick 5 diverse examples
import random
random.seed(42)
selected = random.sample(pairs, min(5, len(pairs)))
for ori, adv in selected:
    print(f"  {os.path.basename(ori)}")

In [ ]:
n = len(selected)
fig, axes = plt.subplots(n, 6, figsize=(18, 3.5 * n))

if n == 1:
    axes = [axes]

col_labels = ['Original', 'Adversarial', 'Difference ×10',
              'Original hash', 'Adversarial hash', 'Flipped bits (XOR)']

for idx, (ori_path, adv_path) in enumerate(selected):
    visualize_attack(ori_path, adv_path, axes[idx])
    # Left label: image stem
    stem = os.path.basename(ori_path).replace('.JPEG','').replace('.jpeg','')[:20]
    axes[idx][0].set_ylabel(stem, fontsize=7, rotation=0, labelpad=60, va='center')

# Column headers
for ax, label in zip(axes[0], col_labels):
    ax.set_title(label, fontsize=10, fontweight='bold', pad=6)

plt.suptitle(
    'pHash Evasion Attack — Image and Hash Visualization (T=0.10)\n'
    'Blue = bit 1  |  Gray = bit 0  |  Red = flipped bit',
    fontsize=11, y=1.01
)
plt.tight_layout()
out = os.path.join(FIG_DIR, 'phash_attack_visualization.png')
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print("Saved:", out)

## Single-image deep dive — show one example in detail

In [ ]:
ori_path, adv_path = selected[0]
device = torch.device('cpu')

ori_t = load_and_preprocess_img(ori_path, device, resize=True)
adv_t = load_and_preprocess_img(adv_path, device, resize=True)

ori_hash, ori_hex, ori_median = compute_phash_hard(ori_t)
adv_hash, adv_hex, adv_median = compute_phash_hard(adv_t)

bits_flipped = int((ori_hash != adv_hash).sum())
d_norm = bits_flipped / 64.0

print("=" * 50)
print(f"Image: {os.path.basename(ori_path)}")
print("=" * 50)
print(f"  Original hash  : {ori_hex}")
print(f"  Adversarial hash: {adv_hex}")
print(f"  Hashes match?  : {ori_hex == adv_hex}")
print()
print(f"  Bits flipped   : {bits_flipped} / 64")
print(f"  d_norm         : {d_norm:.4f}")
print(f"  Threshold T    : 0.10")
print(f"  Threshold met? : {d_norm >= 0.10}")
print()
print("  Original hash bits (row by row):")
for i in range(8):
    row = ori_hash[i*8:(i+1)*8].int().tolist()
    print(f"  row {i}: {row}")

print()
print("  Flipped bit positions (0-indexed):")
flipped_positions = (ori_hash != adv_hash).nonzero(as_tuple=True)[0].tolist()
print(f"  {flipped_positions}")
print(f"  Grid positions (row, col):\n  ",
      [(p // 8, p % 8) for p in flipped_positions])

## Hash binary string comparison — side by side

In [ ]:
ori_bits = ori_hash.int().tolist()
adv_bits = adv_hash.int().tolist()

print(f"{'Pos':>4} | {'Orig':>4} | {'Adv':>4} | {'Flipped?':>8}")
print("-" * 32)
for i, (o, a) in enumerate(zip(ori_bits, adv_bits)):
    flipped = "← FLIP" if o != a else ""
    if o != a or i % 8 == 0:
        print(f"{i:>4} | {o:>4} | {a:>4} | {flipped}")

print()
print(f"Total flipped: {sum(o != a for o, a in zip(ori_bits, adv_bits))} / 64")
print(f"d_norm = {sum(o != a for o, a in zip(ori_bits, adv_bits))/64:.4f}")

## Dataset threshold transfer — ImageNette DCT coefficient distribution

Addresses the question: *does a threshold calibrated on Dataset A maintain security properties on Dataset B?*  
We check the DCT median distribution across all 500 ImageNette images to understand threshold sensitivity.

In [ ]:
import pandas as pd
from utils.phash_torch import compute_phash_soft

INPUT_DIR = os.path.join(REPO_ROOT, "inputs", "inputs_500")
all_images = sorted(glob.glob(os.path.join(INPUT_DIR, "*")))

medians = []
for img_path in all_images[:500]:
    try:
        t = load_and_preprocess_img(img_path, device, resize=True)
        _, _, m = compute_phash_hard(t)
        medians.append(m)
    except Exception:
        pass

medians = np.array(medians)
print(f"Computed medians for {len(medians)} images")
print(f"  Median of medians : {np.median(medians):.4f}")
print(f"  Std of medians    : {np.std(medians):.4f}")
print(f"  Min / Max         : {medians.min():.4f} / {medians.max():.4f}")
print()
print("Implication for threshold transfer:")
print("  The frozen-median threshold is image-specific (computed fresh per image).")
print("  This means our attack adapts to each image's DCT distribution.")
print("  A fixed global threshold would behave differently across datasets")
print("  depending on whether their DCT coefficient distributions differ.")

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(medians, bins=40, color='#378ADD', edgecolor='white', linewidth=0.3)
ax.axvline(np.median(medians), color='red', lw=1.5, linestyle='--',
           label=f'Median={np.median(medians):.3f}')
ax.set_xlabel('Per-image DCT median threshold')
ax.set_ylabel('Count')
ax.set_title('Distribution of pHash median thresholds — 500 ImageNette images')
ax.legend()
plt.tight_layout()
out = os.path.join(FIG_DIR, 'phash_median_distribution.png')
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print("Saved:", out)